# CESNET-QUIC22 둘러보기

이번 주 목표: 앱 3~4개의 앞 30개 패킷 크기 시퀀스를 그래프로 그려 본다.

- 처음 실행하면 XS(1,000만 플로우)를 `D:/datasets/CESNET-QUIC22/`로 내려받는다. 오래 걸린다.
- 커널은 저장소의 `.venv`를 고른다.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # notebooks/ 에서 src 를 불러오기 위해

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import DIR, IPT, SIZE
from src.data import load_cesnet

plt.rcParams["font.family"] = "Malgun Gothic"  # 한글 글꼴
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
# 둘러보기만 하므로 일부만 불러온다
data = load_cesnet(size="XS", train_size=200_000, test_size=20_000)
data.condition

In [ ]:
counts = pd.Series(np.bincount(data.train.y, minlength=len(data.classes)), index=data.classes).sort_values(ascending=False)
counts.head(20)

In [ ]:
# 플로우가 많은 앱 4개, 앱마다 플로우 20개의 (방향 × 크기) — 위쪽은 클라이언트→서버, 아래쪽은 서버→클라이언트
apps = counts.index[:4]
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True, sharey=True)
for ax, app in zip(axes.flat, apps):
    X = data.train.X[data.train.y == data.classes.index(app)][:20]
    for flow in X:
        ax.plot(flow[DIR] * flow[SIZE], alpha=0.4)
    ax.axhline(0, color="gray", linewidth=0.5)
    ax.set_title(app)
for ax in axes[1]:
    ax.set_xlabel("패킷 순서")
for ax in axes[:, 0]:
    ax.set_ylabel("방향 × 크기 (바이트)")
plt.tight_layout()

In [ ]:
# 패킷 간격(ms) 분포 — 앱마다 다르게 생겼는지
fig, ax = plt.subplots(figsize=(8, 4))
for app in apps:
    X = data.train.X[data.train.y == data.classes.index(app)]
    ipt = X[:, IPT][X[:, DIR] != 0]
    ax.hist(np.log1p(ipt), bins=60, histtype="step", density=True, label=app)
ax.set_xlabel("log(1 + 간격 ms)")
ax.legend()